In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/csiro-biomass/sample_submission.csv
/kaggle/input/csiro-biomass/train.csv
/kaggle/input/csiro-biomass/test.csv
/kaggle/input/csiro-biomass/test/ID1001187975.jpg
/kaggle/input/csiro-biomass/train/ID2099464826.jpg
/kaggle/input/csiro-biomass/train/ID2037861084.jpg
/kaggle/input/csiro-biomass/train/ID1211362607.jpg
/kaggle/input/csiro-biomass/train/ID1853508321.jpg
/kaggle/input/csiro-biomass/train/ID193102215.jpg
/kaggle/input/csiro-biomass/train/ID698608346.jpg
/kaggle/input/csiro-biomass/train/ID1859251563.jpg
/kaggle/input/csiro-biomass/train/ID1880764911.jpg
/kaggle/input/csiro-biomass/train/ID853954911.jpg
/kaggle/input/csiro-biomass/train/ID1403107574.jpg
/kaggle/input/csiro-biomass/train/ID1781353117.jpg
/kaggle/input/csiro-biomass/train/ID384648061.jpg
/kaggle/input/csiro-biomass/train/ID1563418511.jpg
/kaggle/input/csiro-biomass/train/ID2125100696.jpg
/kaggle/input/csiro-biomass/train/ID482555369.jpg
/kaggle/input/csiro-biomass/train/ID638711343.jpg
/kaggle/input/c

In [8]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms


In [9]:
train_df = pd.read_csv("/kaggle/input/csiro-biomass/train.csv")


In [10]:
TARGETS = [
    "Dry_Clover_g",
    "Dry_Dead_g",
    "Dry_Green_g",
    "GDM_g",
    "Dry_Total_g"
]

pivot_df = (
    train_df
    .pivot(index="image_path", columns="target_name", values="target")
    .reset_index()
)


In [11]:
class BiomassDataset(Dataset):
    def __init__(self, df, img_root, transform=None):
        self.df = df
        self.img_root = img_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, row["image_path"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        targets = row[TARGETS].values.astype(np.float32)
        targets = np.log1p(targets)  # log-transform

        return image, torch.tensor(targets)


In [12]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [13]:
dataset = BiomassDataset(
    pivot_df,
    img_root="/kaggle/input/csiro-biomass",
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2
)


In [14]:
class BiomassCNN_MultiKernel(nn.Module):
    def __init__(self):
        super().__init__()

        # Backbone (NO pretrained weights)
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Identity()  # (B, 512)

        # Expand to pseudo feature map
        self.expand = nn.Unflatten(1, (512, 1, 1))

        # Multi-kernel feature extractors
        self.conv3 = nn.Conv2d(512, 128, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(512, 128, kernel_size=5, padding=2)
        self.conv7 = nn.Conv2d(512, 128, kernel_size=7, padding=3)

        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Linear(128 * 3, 5)

    def forward(self, x):
        x = self.backbone(x)          # (B, 512)
        x = self.expand(x)            # (B, 512, 1, 1)

        f3 = self.relu(self.conv3(x))
        f5 = self.relu(self.conv5(x))
        f7 = self.relu(self.conv7(x))

        x = torch.cat([f3, f5, f7], dim=1)
        x = self.pool(x).squeeze(-1).squeeze(-1)

        return self.head(x)


In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


In [17]:
model = BiomassCNN_MultiKernel().to(device)

for param in model.backbone.parameters():
    param.requires_grad = False


In [18]:
criterion = nn.SmoothL1Loss()  # better than MSE for biomass
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4
)


In [19]:
max_epochs = 20
patience = 3
best_loss = float("inf")
epochs_no_improve = 0
best_state = None


In [20]:
model.train()

for epoch in range(max_epochs):
    total_loss = 0.0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        preds = model(images)
        loss = criterion(preds, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

    if best_loss - avg_loss > 1e-4:
        best_loss = avg_loss
        best_state = model.state_dict()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patience:
        print("Early stopping triggered")
        break


Epoch 1 | Loss: 0.9354
Epoch 2 | Loss: 0.4592
Epoch 3 | Loss: 0.4291
Epoch 4 | Loss: 0.4240
Epoch 5 | Loss: 0.3981
Epoch 6 | Loss: 0.3984
Epoch 7 | Loss: 0.3967
Epoch 8 | Loss: 0.3932
Epoch 9 | Loss: 0.3875
Epoch 10 | Loss: 0.3781
Epoch 11 | Loss: 0.3810
Epoch 12 | Loss: 0.3794
Epoch 13 | Loss: 0.3826
Early stopping triggered


In [21]:
model.load_state_dict(best_state)


<All keys matched successfully>

In [22]:
for param in model.backbone.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [23]:
test_df = pd.read_csv("/kaggle/input/csiro-biomass/test.csv")

target2idx = {t: i for i, t in enumerate(TARGETS)}

model.eval()
predictions = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        img_path = os.path.join(
            "/kaggle/input/csiro-biomass",
            row["image_path"]
        )

        image = Image.open(img_path).convert("RGB")
        image = transform(image).unsqueeze(0).to(device)

        output = model(image).cpu().numpy()[0]
        pred = np.expm1(output[target2idx[row["target_name"]]])

        predictions.append(pred)


In [24]:
submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "target": predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv saved successfully")
submission.head()


submission.csv saved successfully


,sample_id,target
0,ID1001187975__Dry_Clover_g,2.206732
1,ID1001187975__Dry_Dead_g,8.692155
2,ID1001187975__Dry_Green_g,16.225378
3,ID1001187975__Dry_Total_g,43.682140
4,ID1001187975__GDM_g,23.041981
